# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KhanBuilds/Rayanflyrank/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

**Lane 2: Refresh / Content Opportunity Scoring** — ranking mature content for editorial refresh review.

**Status (2026-07-30): partial.** Sections 1–3, 5 and 7 are complete and reproducible. Section 4 carries
the **baseline** result only; the model row is **PENDING ML-08**, and Section 6's queue is therefore the
baseline's, not the model's. Nothing in this notebook reports a model number I have not run — where a
number is missing, it says PENDING rather than borrowing one.

Written work lives in `work/capstone_report.md`; the assignment notebooks it draws on are
`w01_research_question.ipynb` (ML-02), `w02_ml_task_framing.ipynb` (ML-03),
`w03_data_contract.ipynb` (ML-04) and `w04_baseline_score.ipynb` (ML-07).

In [1]:
# Setup: work from the repo root so the starter CSV loads on Colab AND from a fresh local clone.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/KhanBuilds/Rayanflyrank"
REPO_DIR = "Rayanflyrank"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != os.path.dirname(os.getcwd()):
        os.chdir("..")

DATA_PATH = "data/raw/content_refresh_anonymized.csv"
assert os.path.exists(DATA_PATH), f"starter CSV not found at {DATA_PATH} - are you at the repo root?"

import numpy as np
import pandas as pd

RANDOM_STATE = 42

df = pd.read_csv(DATA_PATH)
# The label is DEFINED from trend_direction, which is computed from the 30-day impression pair.
# So trend_direction, trend_pct, impressions_last_30d and impressions_prev_30d are never features.
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
y = df["is_declining_label"].to_numpy()
BASE_RATE = float(y.mean())


def precision_at_k(labels, scores, k: int) -> float:
    """Share of genuine positives among the k highest-scoring rows."""
    order = np.argsort(-np.asarray(scores, dtype=float), kind="stable")
    return float(np.asarray(labels)[order[:k]].mean())


print("Working dir:", os.getcwd())
print(f"Loaded {len(df):,} rows x {df.shape[1]} columns; {df['client_id'].nunique()} clients.")
print(f"Label base rate: {BASE_RATE:.4f}")


Working dir: /content/Rayanflyrank
Loaded 30,000 rows x 45 columns; 32 clients.
Label base rate: 0.5421


## 1. Question

**Among mature indexed content items with established search demand, which specific pages are
undergoing measured organic decline and should be prioritised for editorial review in the coming
sprint?**

**The decision it supports.** A FlyRank content strategist has roughly 20–50 review slots per sprint
against a portfolio of tens of thousands of published pages. The binding constraint is editorial
capacity, so the useful output is an *ordering*, not a verdict on every page.

- **Unit of analysis:** one pseudonymized content item (`content_id`).
- **Output:** a priority score per page, delivered as a ranked queue with reason codes.
- **Action:** the editor opens the top K pages and either refreshes (update facts, realign headers with
  current intent, expand thin sections, refresh metadata) or, having looked, deliberately skips.
- **Cost of a wrong call:** a false positive burns a review slot worth roughly $300–$1,000 of editorial
  time on a stable page; a false negative leaves a decaying revenue page unreviewed while competitors
  take the position. The costs are asymmetric and capacity is fixed, which is why **precision at the top
  of the ranking** is the metric and accuracy is not.

**Task type:** ranking / scoring. A classifier is fitted underneath, but its probability is used as a
ranking score and evaluated with precision@K — never thresholded at 0.5 and reported as accuracy.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Section 1: the decision this output feeds is capacity-limited - that is what picks the metric.
CAPACITY_PER_SPRINT = 50
print(f"portfolio in this slice      {len(df):,} pages")
print(f"review slots per sprint      {CAPACITY_PER_SPRINT}")
print(f"share of portfolio acted on  {CAPACITY_PER_SPRINT / len(df):.2%}")
print(f"label base rate              {BASE_RATE:.4f}  <- every precision number sits next to this")

portfolio in this slice      30,000 pages
review slots per sprint      50
share of portfolio acted on  0.17%
label base rate              0.5421  <- every precision number sits next to this


## 2. Data

**Release used:** the anonymized starter dataset that ships in this repo —
`data/raw/content_refresh_anonymized.csv`, **30,000 rows × 44 columns**, one row per pseudonymized
content item, **32 pseudonymized clients**, trailing-90-day metrics. The gated warehouse release
(`hf://datasets/FlyRank/internship-warehouse`) was **not** used; where it would change a conclusion, that
is stated in Limitations.

**Date window.** The file contains **no date column at all** — verified, not assumed. Every time field is
a relative offset from an unstated snapshot date. Three window facts that follow:

- The two 30-day columns cover days 1–30 and 31–60; days 61–90 sit in the 90-day totals and in neither.
  `last_30d + prev_30d == impressions_90d` holds for only **8.7%** of rows.
- `days_with_impressions` caps at **88** while `days_with_sessions` reaches **90** — the GSC reporting lag,
  visible in the data. Search and analytics windows are not the same length.
- `content_age_days` has a minimum of exactly **90**: the file was pre-filtered upstream to mature pages,
  so the 30,000-row count is the file as shipped, not the product of my own filter.

**Excluded, and why.**

| Excluded | Why |
|---|---|
| `trend_direction`, `trend_pct` | The label source (`is_declining_label = trend_direction == "down"`). |
| `impressions_last_30d`, `impressions_prev_30d` | **Measured to reconstruct the label exactly (1.0000).** The label is a deterministic function of this pair, so they are leakage — a stronger exclusion than the docs' two named fields. |
| `clicks_last_30d/prev_30d`, `sessions_last_30d/prev_30d` | Same last-vs-prev ratio shape. Measured agreement with the label is only 0.536 / 0.538, so not leaks — dropped anyway as weak and hard to defend. |
| `provider_used`, `model_used` | Generation-provenance / product-decision flags (71.5% and 19.1% missing). They describe which internal pipeline wrote the page, not whether it is decaying. |
| `content_id`, `client_id` | Pseudonymous IDs — context only: grouping, joining, splitting, audit. Never features. |

That leaves **32 feature fields**. `w03_data_contract.ipynb` asserts in code that all 44 file columns plus
the engineered label land in exactly one bucket, so the contract cannot drift from the file.

**Public-safety.** No client names, domains, URLs or raw search queries appear anywhere in `work/`. IDs
are pseudonyms. Printouts in these notebooks show aggregate counts, and the top-20 review withholds
identifiers. The ranked queue CSV stays gitignored (`work/**/*.csv`); only metrics JSONs are committed.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Section 2: the data-safety claims above, re-checked here so the paper's numbers are self-verifying.
LABEL_DERIVED = ["trend_direction", "trend_pct", "impressions_last_30d", "impressions_prev_30d"]
CONTEXT = ["content_id", "client_id"]
EXCLUDED = ["clicks_last_30d", "clicks_prev_30d", "sessions_last_30d", "sessions_prev_30d",
            "provider_used", "model_used"]
FEATURES = [c for c in df.columns if c not in LABEL_DERIVED + CONTEXT + EXCLUDED + ["is_declining_label"]]

print(f"file columns {df.shape[1] - 1} + engineered label | features {len(FEATURES)} | "
      f"label-derived {len(LABEL_DERIVED)} | context {len(CONTEXT)} | excluded {len(EXCLUDED)}")
assert not set(FEATURES) & set(LABEL_DERIVED + CONTEXT + EXCLUDED), "bucket overlap"

# The leakage measurement that justifies the strongest exclusion.
def reconstruct(last, prev):
    pct = np.where(prev > 0, (last - prev) / prev.where(prev > 0) * 100.0, np.nan)
    return np.where(prev == 0, 0, np.where(pct < -20.0, 1, 0))

print()
for metric in ("impressions", "clicks", "sessions"):
    agree = (reconstruct(df[f"{metric}_last_30d"], df[f"{metric}_prev_30d"]) == y).mean()
    verdict = "LEAK - excluded" if agree > 0.99 else "near base rate - not a leak"
    print(f"  {metric + ' 30d pair':22s} agreement {agree:.4f}  {verdict}")

print()
print("Window facts:")
tiles = (df["impressions_last_30d"] + df["impressions_prev_30d"] == df["impressions_90d"]).mean()
print(f"  30d columns tile the 90d window in {tiles:.1%} of rows")
print(f"  days_with_impressions max {df['days_with_impressions'].max()} (GSC) vs "
      f"days_with_sessions max {df['days_with_sessions'].max()} (GA4)")
print(f"  content_age_days min {df['content_age_days'].min()} -> file pre-filtered to mature pages")
print()
print("Public-safety scan of the columns this project reads:")
UNSAFE_HINTS = ("url", "domain", "title", "query", "keyword_text", "client_name", "slug")
hits = [c for c in df.columns if any(h in c.lower() for h in UNSAFE_HINTS)]
print(f"  columns that could carry identifying text: {hits or 'none'}")

file columns 44 + engineered label | features 32 | label-derived 4 | context 2 | excluded 6

  impressions 30d pair   agreement 1.0000  LEAK - excluded
  clicks 30d pair        agreement 0.5364  near base rate - not a leak
  sessions 30d pair      agreement 0.5383  near base rate - not a leak

Window facts:
  30d columns tile the 90d window in 8.7% of rows
  days_with_impressions max 88 (GSC) vs days_with_sessions max 90 (GA4)
  content_age_days min 90 -> file pre-filtered to mature pages

Public-safety scan of the columns this project reads:
  columns that could carry identifying text: none


## 3. Methodology

**Assumptions, stated so they can be attacked.**

1. The dataset's `trend_direction` is an acceptable *proxy* for "in measured decline". It is not an
   observed editorial outcome and not a recovery measurement.
2. A page's trailing-90-day performance shape carries information about whether it is decaying.
3. Editorial capacity is fixed and small, so the top of the ranking is the only part that matters.

**Label.** `is_declining_label = 1` when `trend_direction == "down"`, i.e. impressions fell more than 20%
between the most recent 30 days and the 30 before that. Base rate **0.5421** (16,262 of 30,000).

Two properties of this label that shape every claim downstream:

- It is **defined, not observed** — a threshold on a ratio, so the model learns the dataset's definition
  of decline rather than the world's.
- It is **noise-sensitive at low volume**: 19.1% of declining pages have fewer than 100 impressions in
  90 days, where 5 → 3 impressions reads as "down 40%". And it **cannot fire at all** for the 3,388 rows
  with `impressions_prev_30d == 0`.

**Features.** The 32 contract-approved fields: demand/market (`search_volume`, `competition`,
`competition_level`, `cpc`), page shape (`word_count`, `char_count`, tiers, `content_type`, `main_intent`),
90-day totals (impressions, clicks, pageviews, sessions, users, engaged sessions, AI sessions, scroll
events), coverage (`days_with_impressions`, `days_with_sessions`), age/freshness (`content_age_days`,
`age_tier`, `age_tier_order`, `days_since_last_update`, `freshness_tier`), and rates/position (`ctr`,
`avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `impression_tier`, `position_tier`).

Handling that is not optional here: **`avg_position == 0` means "no data"** (1,205 rows) so it becomes
missing plus a flag, never a zero; and **missingness follows `content_type`** — `feedly article` is 100%
missing `search_volume`/`competition`/`cpc`, `keyword article` 28.3% missing `word_count` — while the label
rate differs sharply by type (28.7% vs 56.1% vs 57.2%). A blind `fillna(0)` would hand the model a proxy
for "this is a feedly article" disguised as a demand feature, so missing values get explicit `has_*`
indicator flags.

**Baseline (built, ML-07).** A transparent five-condition additive score, no fitted weights:
`3×established_coverage + 2×has_demand + 2×mid_position + 1×stale_90d + 1×mature_page`, with reason codes
on every row. Disclosed weakness: its thresholds were read off the ML-03 label-rate crosstabs, so it is
**mildly tuned in-sample** and its numbers here are optimistic.

**Validation design.** **Grouped by `client_id`** — no client may appear in both train and test. Per-client
label rates span **0.000 to 0.937** across 32 clients, and the three largest clients hold **43.3%** of all
rows, so a random row split would measure client memorisation rather than generalisation. A **time-aware**
split is impossible from this file (no date column); that needs the warehouse release. Seed 42 throughout.

**Leakage checks run.** (a) The label-reconstruction test above, which produced the exclusion of the
30-day impression pair. (b) A code assertion that the baseline's five inputs contain no label-derived or
ID column. (c) An AUC sanity check — the baseline's ROC-AUC is **0.6485**, and a near-1.0 value from five
hand-written conditions would have been the alarm. (d) The contract's bucket-overlap assertions.

**PENDING (ML-08/ML-09):** the model itself, its held-out comparison against this baseline on the same
client-grouped split, and a permutation/feature-importance read. `scikit-learn` is not installed in the
environment these notebooks were last executed in, which is why no model number appears below.

In [4]:
# Rebuild the ML-07 baseline inline, so this notebook stands alone (no dependency on
# another notebook having been run first). Then cross-check against the committed metrics file.
import json
from pathlib import Path

position = df["avg_position"].replace(0, np.nan)  # 0 means "no data", not rank 0
RULE = {
    "established_coverage": (df["days_with_impressions"].between(20, 87), 3),
    "has_demand":           (df["impressions_90d"] >= 40, 2),
    "mid_position":         (((position > 3) & (position <= 50)).fillna(False), 2),
    "stale_90d":            (df["days_since_last_update"] >= 90, 1),
    "mature_page":          (df["content_age_days"].between(90, 364), 1),
}
baseline = df[["content_id", "client_id", "is_declining_label", "impressions_90d",
               "days_with_impressions", "avg_position", "ctr", "days_since_last_update",
               "content_age_days", "content_type"]].copy()
baseline["baseline_score"] = sum(flag.astype(int) * pts for flag, pts in RULE.values())
flag_frame = pd.DataFrame({name: flag.to_numpy() for name, (flag, _) in RULE.items()})
baseline["reason_codes"] = [",".join(flag_frame.columns[row]) or "no_signal" for row in flag_frame.to_numpy()]
baseline["rank_key"] = baseline["baseline_score"] * 100 + np.minimum(np.log1p(baseline["impressions_90d"]), 10)

queue = baseline.sort_values("rank_key", ascending=False, kind="stable").reset_index(drop=True)
queue.insert(0, "queue_rank", np.arange(1, len(queue) + 1))
y_queue = queue["is_declining_label"].to_numpy()

K_VALUES = [20, 50, 100, 200, 500]
baseline_metrics = {f"precision_at_{k}": round(float(y_queue[:k].mean()), 4) for k in K_VALUES}
print(f"base rate {BASE_RATE:.4f}")
for k in K_VALUES:
    print(f"  baseline precision@{k:<4d} {baseline_metrics[f'precision_at_{k}']:.4f}")

# Reproducibility check: do these match the receipt committed by w04_baseline_score.ipynb?
receipt = Path("work/outputs/baseline_metrics.json")
if receipt.exists():
    committed = json.loads(receipt.read_text())
    same = all(committed[k] == v for k, v in baseline_metrics.items())
    print(f"\nmatches work/outputs/baseline_metrics.json: {same}")
    assert same, "this notebook and the committed receipt disagree - one of them is stale"
else:
    print("\nwork/outputs/baseline_metrics.json not found - run w04_baseline_score.ipynb to write it")


base rate 0.5421
  baseline precision@20   0.7500
  baseline precision@50   0.7400
  baseline precision@100  0.8000
  baseline precision@200  0.8350
  baseline precision@500  0.8080

matches work/outputs/baseline_metrics.json: True


## 4. Results (vs baseline)

**The honest table.** One row is real; one is deliberately empty.

| System | Split | Precision@20 | Precision@50 | Precision@100 | ROC-AUC | Base rate |
|---|---|---|---|---|---|---|
| Flag everything (floor) | — | 0.542 | 0.542 | 0.542 | 0.500 | 0.542 |
| **My rule baseline (ML-07)** | full data, in-sample | **0.750** | **0.740** | **0.800** | **0.649** | 0.542 |
| My model (ML-08) | client-held-out | *PENDING* | *PENDING* | *PENDING* | *PENDING* | 0.542 |
| *Reference pipeline, random forest* | *client holdout* | *—* | *0.740* | *—* | *0.750* | *0.542* |
| *Reference pipeline, rule baseline* | *full data* | *—* | *0.240* | *—* | *0.627* | *0.542* |

The two italic rows are **the repo's bundled results** (`outputs/model_report.md`), not mine. They are the
bar to clear, not a result I can claim.

**What the baseline row means.** Of 50 review slots, about **37** land on pages measured as declining,
against about **27** for random triage — a 1.37× lift on the metric that matches the decision. It clears
the ≥0.70 "useful" threshold I committed to in ML-03 before building anything.

**Three caveats that belong in the same breath as the number:**

1. **It is in-sample.** The rule's thresholds came from looking at label rates in this same data. A
   held-out estimate would be lower. The committed receipt
   (`work/outputs/baseline_metrics.json`) says so in its `evaluation` field.
2. **Precision rises with K** — 0.750 @20, 0.740 @50, 0.800 @100, 0.835 @200. A ranking that improves
   deeper into the list is mis-ordered at the top. Diagnosis: within the max-score band I break ties by
   impressions, and the biggest pages here are *more* stable, so the three highest-ranked pages in my
   queue all measured `up` (+35%, +54%, +69%). The rule finds a good **band** (2,990 pages at score 9,
   71.4% declining) but does not rank *within* it.
3. **The score is not monotone**: score 7 (0.467) sits below score 6 (0.593), and score 1 (0.088) below
   score 0 (0.293, n=75). Five added opinions are not a calibrated ordering.

**Why this is the interesting result rather than a disappointing one.** The gap the baseline leaves is
specific and stated in advance: *good band, bad ordering inside it*. That is exactly what a fitted model
should be able to fix, and it gives ML-08 a falsifiable prediction to test rather than a vague hope of
"beating the baseline".

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Section 4: the results table, assembled from measured numbers only. No model row is invented.
def roc_auc(labels, scores) -> float:
    """AUC via the rank-sum identity - no sklearn needed."""
    labels = np.asarray(labels)
    ranks = pd.Series(scores).rank().to_numpy()
    n_pos, n_neg = labels.sum(), (1 - labels).sum()
    return float((ranks[labels == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))

baseline_auc = roc_auc(y_queue, queue["baseline_score"])

rows = [
    {"system": "flag everything (floor)", "split": "-",
     **{f"p@{k}": round(BASE_RATE, 3) for k in (20, 50, 100)}, "roc_auc": 0.500},
    {"system": "my rule baseline (ML-07)", "split": "full data, in-sample",
     **{f"p@{k}": baseline_metrics[f"precision_at_{k}"] for k in (20, 50, 100)},
     "roc_auc": round(baseline_auc, 4)},
    {"system": "my model (ML-08)", "split": "client-held-out",
     **{f"p@{k}": None for k in (20, 50, 100)}, "roc_auc": None},
]
results = pd.DataFrame(rows).set_index("system")
print(results.to_string(na_rep="PENDING"))
print(f"\nbase rate {BASE_RATE:.4f} | lift at K=50: "
      f"{baseline_metrics['precision_at_50'] / BASE_RATE:.2f}x")
print(f"slots landing on declining pages at K=50: "
      f"{baseline_metrics['precision_at_50'] * 50:.0f}/50 vs {BASE_RATE * 50:.0f}/50 for random triage")
print()
print("Caveat 2 in numbers - precision RISES with K, so the top of the ranking is mis-ordered:")
for k in K_VALUES:
    print(f"  precision@{k:<4d} {baseline_metrics[f'precision_at_{k}']:.4f}")
print()
print("The three highest-ranked pages, and what they actually did:")
top3 = queue.head(3).merge(df[["content_id", "trend_direction", "trend_pct"]], on="content_id", how="left")
print(top3[["queue_rank", "impressions_90d", "baseline_score", "trend_direction", "trend_pct"]].to_string(index=False))
print("  -> all three grew. The impressions tie-break, not the rule, put them at the top.")
print()
print(f"baseline ROC-AUC {baseline_auc:.4f} (modest, as a five-condition rule should be - no leakage)")
assert baseline_auc < 0.90, "suspiciously high AUC for a hand-written rule - check for leakage"

                                         split    p@20    p@50   p@100  roc_auc
system                                                                         
flag everything (floor)                      -   0.542   0.542   0.542   0.5000
my rule baseline (ML-07)  full data, in-sample   0.750   0.740   0.800   0.6485
my model (ML-08)               client-held-out PENDING PENDING PENDING  PENDING

base rate 0.5421 | lift at K=50: 1.37x
slots landing on declining pages at K=50: 37/50 vs 27/50 for random triage

Caveat 2 in numbers - precision RISES with K, so the top of the ranking is mis-ordered:
  precision@20   0.7500
  precision@50   0.7400
  precision@100  0.8000
  precision@200  0.8350
  precision@500  0.8080

The three highest-ranked pages, and what they actually did:
 queue_rank  impressions_90d  baseline_score trend_direction  trend_pct
          1            12863               9              up       54.4
          2            10356               9              up       68.5

## 5. Limitations

**What this work cannot claim.**

1. **No causal claim.** Nothing here records an intervention: no page was refreshed because of a score,
   and no outcome was measured afterwards. "Refreshing these pages will recover traffic" is unsupported
   and would need a controlled or matched design. The output is a **review queue**.
2. **No forecast.** With no date column and no future window, every number is concurrent. The honest
   sentence is "this page resembles the pages measured as declining", never "this page will decline".
3. **No claim about search-engine behaviour.** I did not model or reverse-engineer any ranking algorithm.
   I modelled observable performance metrics in one pseudonymized portfolio.
4. **No editorial-quality claim.** The score reads traffic shape, not prose quality, factual accuracy or
   strategic value. A page can be decaying because the topic died, and no rewrite fixes that.
5. **The label is a definition, not an outcome** — and a noisy one at low volume (19.1% of declining
   pages have <100 impressions in 90 days). It also cannot fire for 3,388 rows where prior-30-day
   impressions are zero, which is why 46.6% of `feedly article` rows are structurally unlabelable as
   declining.
6. **Survivorship, twice over.** Pages younger than 90 days and pages with zero traffic were removed
   before I received the file. Any "x% of pages are declining" statement means *of mature pages that
   still get some traffic*.
7. **The panel is unbalanced.** 32 clients, 3 to 7,008 pages each, per-client label rates 0.000–0.937.
   Portfolio-wide averages are dominated by a few large clients.
8. **The baseline's numbers are in-sample** (thresholds read off this data's crosstabs) and therefore
   optimistic.
9. **1,205 pages are partly blind** — `avg_position == 0` means no position reading, not rank 1.
10. **Operational limit the metric cannot see:** the top-50 queue draws on only **8 of 32 clients**, with
    one client supplying **44%** of it. A production queue needs a per-client cap — a product decision,
    not a modelling one.
11. **Not yet evaluated out-of-sample at all.** No model, no held-out split, no sealed test. Sections 4
    and 6 are baseline-only, and I make no sealed-evaluation claim anywhere.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Section 5: the limits, quantified rather than asserted.
declining = df[df["trend_direction"] == "down"]
print(f"noise-sensitive label: {(declining['impressions_90d'] < 100).mean():.1%} of declining pages "
      f"have <100 impressions in 90 days (n={(declining['impressions_90d'] < 100).sum():,})")
print(f"label cannot fire:     {(df['impressions_prev_30d'] == 0).sum():,} rows (prev-30d impressions == 0)")
print(f"partly blind:          {(df['avg_position'] == 0).sum():,} rows with avg_position == 0")
print(f"survivorship:          min content_age_days {df['content_age_days'].min()}, "
      f"rows with zero 90d impressions {(df['impressions_90d'] == 0).sum()}")

per_client = df.groupby("client_id").agg(pages=("content_id", "size"), rate=("is_declining_label", "mean"))
print(f"panel imbalance:       {len(per_client)} clients, {per_client['pages'].min()}-{per_client['pages'].max():,} "
      f"pages each, label rate {per_client['rate'].min():.3f}-{per_client['rate'].max():.3f}")
print(f"                       top 3 clients hold {per_client['pages'].nlargest(3).sum() / len(df):.1%} of rows")

counts = queue.head(50)["client_id"].value_counts()
print(f"queue concentration:   top 50 spans {counts.size} of {len(per_client)} clients; "
      f"largest single client = {counts.iloc[0] / 50:.0%}")

noise-sensitive label: 19.1% of declining pages have <100 impressions in 90 days (n=3,110)
label cannot fire:     3,388 rows (prev-30d impressions == 0)
partly blind:          1,205 rows with avg_position == 0
survivorship:          min content_age_days 90, rows with zero 90d impressions 0
panel imbalance:       32 clients, 3-7,008 pages each, label rate 0.000-0.937
                       top 3 clients hold 43.3% of rows
queue concentration:   top 50 spans 8 of 32 clients; largest single client = 44%


## 6. Ranked recommendations

**What a FlyRank editor would do tomorrow, from the baseline queue.** This is the ML-07 output; the
model-ranked version is PENDING ML-08.

The queue maps reason codes to a recommended action. Every row is a **review** recommendation — the
system never recommends publishing a change, only looking at a page.

| Reason-code pattern | Recommended action | Confidence | Honest caveat |
|---|---|---|---|
| All five codes fired, mid-size page | **Refresh review** — check intent alignment, update facts, expand thin sections | Medium-high (~71% of this band measured declining) | Off-season intent looks identical; no seasonality field exists to rule it out |
| All five codes, **largest pages in the band** | **Monitor only, do not spend a slot** | Low — measured counter-signal | The three biggest pages in my top band all *grew*; size is anti-correlated with decline here |
| `established_coverage` + `has_demand`, no `stale_90d` | **Monitor** | Low-medium | Recently updated; a second refresh is unlikely to be the lever |
| Missing `mid_position` (top-3 or no position data) | **Deprioritise** | Medium | `top_3` pages decline least (24.1%); `avg_position == 0` means no reading at all |
| `no_signal` | **Leave alone** | — | 75 pages, and their measured decline rate (0.293) is below the base rate |

**How to run the sprint, concretely:**

1. Take the top 50 rows of `work/outputs/baseline_action_score.csv`.
2. **Drop ranks 1–3** (or any page in the top decile of impressions within the band) into a monitor list
   instead — that is where the measured false positives concentrate.
3. **Cap at ~8 pages per client** so the queue does not spend 44% of a sprint on one portfolio.
4. Review the remainder by hand; the reason codes tell the editor what to look at first.
5. Log the decision (refreshed / skipped / monitored) **with a date**. That log is the observed outcome
   this project currently lacks, and it is what would make a real past→future label possible next quarter.

**Confidence, stated plainly.** Directional and decision-support. The queue is measurably better than
random triage at the metric that matches the decision (0.740 vs 0.542 at K=50), in-sample, on one
90-day snapshot of 32 pseudonymized clients. It is not validated out-of-sample yet, it is not a forecast,
and it does not establish that refreshing anything recovers traffic.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Section 6: the actual recommendation output an editor would receive.
ACTION_BY_SCORE = {9: "refresh_review", 8: "refresh_review", 7: "monitor", 6: "monitor"}

sprint = queue.head(50).copy()
sprint["action"] = sprint["baseline_score"].map(ACTION_BY_SCORE).fillna("monitor")

# Recommendation 2: demote the largest pages in the band - that is where the misses concentrate.
size_cut = sprint["impressions_90d"].quantile(0.90)
sprint.loc[sprint["impressions_90d"] >= size_cut, "action"] = "monitor_only_large_page"

# Recommendation 3: cap per client so one portfolio cannot own the sprint.
PER_CLIENT_CAP = 8
sprint["rank_in_client"] = sprint.groupby("client_id").cumcount() + 1
sprint.loc[sprint["rank_in_client"] > PER_CLIENT_CAP, "action"] = "deferred_client_cap"

print("Sprint queue composition after the two operational rules:")
print(sprint["action"].value_counts().to_string())
print()
sendable = sprint[sprint["action"] == "refresh_review"]
print(f"pages actually sent to an editor: {len(sendable)}")
print(f"  of those, measured declining:   {sendable['is_declining_label'].mean():.3f} "
      f"(base rate {BASE_RATE:.3f})")
print(f"  distinct clients represented:   {sendable['client_id'].nunique()}")
print()
print("Reason-code patterns in the sendable set (counts only, no identifiers):")
print(sendable["reason_codes"].value_counts().head().to_string())
print()
print("NOTE: the raw top-50 precision is the honest headline number "
      f"({baseline_metrics['precision_at_50']:.3f}); the filtered set above is a product decision "
      "layered on top, not a better model.")

Sprint queue composition after the two operational rules:
action
refresh_review             23
deferred_client_cap        22
monitor_only_large_page     5

pages actually sent to an editor: 23
  of those, measured declining:   0.870 (base rate 0.542)
  distinct clients represented:   8

Reason-code patterns in the sendable set (counts only, no identifiers):
reason_codes
established_coverage,has_demand,mid_position,stale_90d,mature_page    23

NOTE: the raw top-50 precision is the honest headline number (0.740); the filtered set above is a product decision layered on top, not a better model.


## 7. Artifacts the paper embeds

Two figures, written to `work/figures/` as SVG, plus their table views — because a figure should never
be the only way to read a number.

- **Figure 1 — `fig1_label_rate_by_score.svg`:** measured decline rate by baseline score level, with the
  base rate drawn as a reference line. This is the chart that shows both the signal (0.088 → 0.714) and
  the honest defect (the score is not monotone).
- **Figure 2 — `fig2_precision_at_k.svg`:** precision@K against the base rate. This is the chart that
  shows the ordering problem — the curve rises with K instead of falling.

Both are single-series charts on a light surface, using slot 1 of the project's validated palette. The
base rate appears as a labelled reference line in both, so no precision number is ever displayed without
it.

A third figure — model vs baseline on the held-out split — is **PENDING ML-08**.

In [8]:
# Two figures the paper embeds. Written to work/figures/ as SVG.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# Colors: slot 1 of the validated reference palette, on a light chart surface.
SERIES_1 = "#2a78d6"
SURFACE = "#fcfcfb"
INK = "#0b0b0b"
INK_MUTED = "#52514e"
GRID = "#e3e2dd"

fig_dir = Path("work/figures")
fig_dir.mkdir(parents=True, exist_ok=True)


def style_axes(ax):
    """Recessive grid and axes; the data carries the chart."""
    ax.set_facecolor(SURFACE)
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID)
    ax.tick_params(colors=INK_MUTED, labelsize=9, length=0)


# --- Figure 1: label rate by rule score level -----------------------------
by_score = baseline.groupby("baseline_score")["is_declining_label"].agg(["size", "mean"])

fig, ax = plt.subplots(figsize=(7.2, 4.0), facecolor=SURFACE)
style_axes(ax)
bars = ax.bar(by_score.index, by_score["mean"], width=0.62, color=SERIES_1)
ax.axhline(BASE_RATE, color=INK_MUTED, linewidth=1.4, linestyle=(0, (5, 4)))
ax.annotate(f"base rate {BASE_RATE:.3f}\n(flag everything)",
            xy=(0.4, BASE_RATE), xytext=(0.4, BASE_RATE + 0.06),
            color=INK_MUTED, fontsize=8.5, va="bottom")

top = by_score["mean"].idxmax()
ax.annotate(f"{by_score.loc[top, 'mean']:.3f}  (n={int(by_score.loc[top, 'size']):,})",
            xy=(top, by_score.loc[top, "mean"]), xytext=(0, 6), textcoords="offset points",
            ha="center", color=INK, fontsize=9, fontweight="bold")

ax.set_title("Measured decline rate rises with the rule's score - but not monotonically",
             color=INK, fontsize=11.5, loc="left", pad=12)
ax.set_xlabel("Baseline rule score (0-9)", color=INK_MUTED, fontsize=9.5)
ax.set_ylabel("Share measured as declining", color=INK_MUTED, fontsize=9.5)
ax.set_xticks(by_score.index)
ax.set_ylim(0, 0.85)
fig.tight_layout()
fig.savefig(fig_dir / "fig1_label_rate_by_score.svg", format="svg", facecolor=SURFACE)
plt.close(fig)

# --- Figure 2: precision@K vs the base rate -------------------------------
fig, ax = plt.subplots(figsize=(7.2, 4.0), facecolor=SURFACE)
style_axes(ax)
ks = K_VALUES
ps = [baseline_metrics[f"precision_at_{k}"] for k in ks]
ax.plot(ks, ps, color=SERIES_1, linewidth=2.0, marker="o", markersize=8,
        markeredgecolor=SURFACE, markeredgewidth=2)
ax.axhline(BASE_RATE, color=INK_MUTED, linewidth=1.4, linestyle=(0, (5, 4)))
ax.annotate(f"base rate {BASE_RATE:.3f}", xy=(ks[0], BASE_RATE), xytext=(0, -14),
            textcoords="offset points", color=INK_MUTED, fontsize=8.5)
ax.annotate(f"{ps[0]:.3f}", xy=(ks[0], ps[0]), xytext=(0, 10), textcoords="offset points",
            ha="center", color=INK, fontsize=9, fontweight="bold")
ax.annotate(f"{max(ps):.3f}", xy=(ks[int(np.argmax(ps))], max(ps)), xytext=(0, 10),
            textcoords="offset points", ha="center", color=INK, fontsize=9, fontweight="bold")

ax.set_xscale("log")
ax.set_xticks(ks)
ax.set_xticklabels([str(k) for k in ks])
ax.set_title("Baseline precision@K climbs with K - the ordering inside the top band is weak",
             color=INK, fontsize=11.5, loc="left", pad=12)
ax.set_xlabel("K (editorial capacity, pages reviewed)", color=INK_MUTED, fontsize=9.5)
ax.set_ylabel("Precision@K", color=INK_MUTED, fontsize=9.5)
ax.set_ylim(0.4, 0.95)
fig.tight_layout()
fig.savefig(fig_dir / "fig2_precision_at_k.svg", format="svg", facecolor=SURFACE)
plt.close(fig)

print("wrote work/figures/fig1_label_rate_by_score.svg")
print("wrote work/figures/fig2_precision_at_k.svg")
print()

# The table view, so the figures are never the only way to read the numbers.
table = by_score.rename(columns={"size": "pages", "mean": "declining_rate"}).round(3)
print("Table view of Figure 1:")
print(table.to_string())
print()
print("Table view of Figure 2:")
print(pd.DataFrame({"K": ks, "precision_at_K": ps, "base_rate": BASE_RATE}).to_string(index=False))


wrote work/figures/fig1_label_rate_by_score.svg
wrote work/figures/fig2_precision_at_k.svg

Table view of Figure 1:
                pages  declining_rate
baseline_score                       
0                  75           0.293
1                1809           0.088
2                 612           0.350
3                2785           0.417
4                3356           0.449
5                4331           0.577
6                5145           0.593
7                2606           0.467
8                6291           0.684
9                2990           0.714

Table view of Figure 2:
  K  precision_at_K  base_rate
 20           0.750   0.542067
 50           0.740   0.542067
100           0.800   0.542067
200           0.835   0.542067
500           0.808   0.542067


## ML-12 — Demo, social cut, and employer summary

### 5-minute demo outline

| Time | Beat | What is on screen |
|---|---|---|
| 0:00–0:45 | **The decision, not the model.** An editor has 50 slots and 30,000 pages. Today triage is "anything not touched in six months" — and in this data that rule fires on **17 pages**. | The freshness-tier table: 174 pages at 181+ days |
| 0:45–1:30 | **The metric, chosen before building.** Precision@50 against a 54.2% base rate. Flag-everything already scores 0.542, so any number without its base rate is not a claim. | Figure 2, base-rate line only |
| 1:30–2:30 | **The leakage find.** The label is a −20% threshold on the 30-day impression pair — so those two columns reconstruct it **exactly** (1.0000). The docs name two forbidden fields; the real list is four. | The reconstruction printout: 1.0000 / 0.5364 / 0.5383 |
| 2:30–3:30 | **The baseline.** Five readable conditions, reason codes on every row: 0.740 precision@50 vs 0.542 — 37 of 50 slots instead of 27. | Figure 1 |
| 3:30–4:30 | **The defect I found in my own result.** Precision *rises* with K, and my three top-ranked pages all grew. Good band, bad ordering inside it — which is exactly the gap a model should close. | The top-3 table with `trend_pct` +54%, +69%, +35% |
| 4:30–5:00 | **What it cannot say.** No causation, no forecast, no claim about Google. Plus the operational catch: 44% of the queue is one client. | Limitations slide |

### Social-post cut

> Spent a week ranking 30,000 pages for editorial refresh — and the most useful thing I found was in my
> own numbers.
>
> A transparent 5-condition rule hit **0.740 precision@50** against a **0.542** base rate: 37 of 50
> review slots on genuinely declining pages instead of 27.
>
> Then precision *rose* with K — 0.740 at 50, 0.835 at 200. That only happens if the top of your ranking
> is wrong. The three pages my queue ranked highest had all **grown** 35–69%: my tie-break sorted by
> traffic size, and big pages are the stable ones.
>
> Also worth the week: the label was a −20% threshold on two impression columns, which means those
> columns reconstruct it *exactly*. The docs warned about two fields. The real list was four.
>
> Good band, bad ordering. Next: whether a model can fix the ordering on a client-held-out split.
>
> Built on the FlyRank ML Internship dataset. #MachineLearning #SEO #DataScience

### Employer-facing summary (3 sentences)

> I built a ranked editorial-review queue over 30,000 pseudonymized content items, framing it as a
> capacity-constrained ranking problem and committing to precision@50 against the 54.2% base rate before
> building anything. A transparent five-condition rule reached 0.740 precision@50 in-sample — 37 of 50
> review slots on genuinely declining pages versus 27 for random triage — and I found and documented two
> defects in it: the label was exactly reconstructible from two impression columns (leakage the data
> dictionary did not flag), and precision rose with K, revealing that my own top-ranked pages were the
> growing ones. The work is reproducible from a fresh clone with committed metrics receipts, and every
> claim is scoped to observed/decision-support language — no causal claim, no forecast, and the
> unfinished parts are labelled PENDING rather than filled with borrowed numbers.

## Acknowledgments & data credit

Built on the **FlyRank ML Internship dataset** — [flyrank.ai](https://flyrank.ai). Thanks to the FlyRank
team for the pseudonymized data release and the lane framing.

*(This section is required at the bottom of the deployed paper, alongside the Abstract at the top — see
`work/capstone_report.md` §0 and §9.)*

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled - markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` - then submit your repo URL on the card. Done.
- [x] My deployed paper has **all 9 sections** — Abstract and Acknowledgments live in
      `work/capstone_report.md` (§0 and §9); this notebook mirrors §1–§7.
- [x] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + social-post cut +
      3-sentence employer-facing summary.
- [ ] **Not yet true:** Section 4's model row is PENDING (ML-08), so the paper is not shippable until
      that row is filled or explicitly dropped.